# Python para construir agentes

Este notebook é um curso compacto do Python necessário para a disciplina. Ele não tenta cobrir toda a linguagem: concentra-se nas ideias que reaparecem ao construir modelos, ferramentas, memória e laços de agente. Ao final, você deverá conseguir ler o `agentkit`, representar estado com estruturas nativas, transformar funções em ferramentas por introspecção e escrever um laço robusto e testável.

Os exemplos usam apenas a biblioteca padrão e Python 3.11 ou superior. Nomes permanecem em inglês; comentários e explicações ficam em português; textos que seriam enviados a um modelo ficam em inglês.

## Mapa do curso

O fio condutor será um agente mínimo. Para construí-lo, passaremos por: valores e referências; listas e dicionários; comprehensions e desempacotamento; funções como valores; closures e decoradores; anotações de tipo; exceções; iteradores e geradores; arquivos e JSON; introspecção; testes; e, por fim, o laço completo.

In [ ]:
import inspect
import json
import tempfile
from collections.abc import Callable, Iterable, Iterator
from functools import wraps
from pathlib import Path
from typing import Any, Literal, TypedDict, get_type_hints

## Valores, identidade e mutabilidade

Variáveis são nomes ligados a objetos. Atribuir uma lista a outro nome não a copia: ambos passam a apontar para o mesmo objeto. Esse detalhe é decisivo para históricos e memória, porque uma função pode alterar o estado recebido sem que isso apareça no valor de retorno.

In [ ]:
messages = [{"role": "user", "content": "Hello"}]
same_messages = messages
same_messages.append({"role": "assistant", "content": "Hi"})

print(messages)
print(messages is same_messages)

Uma cópia rasa cria a lista externa, mas compartilha os dicionários internos. Para estado didático, uma estratégia clara é criar novos objetos apenas nos pontos modificados.

In [ ]:
copied = messages.copy()
copied[0]["content"] = "Changed"
print(messages[0])  # o dicionário interno ainda é compartilhado

independent = [message.copy() for message in messages]
independent[0]["content"] = "Independent"
print(messages[0], independent[0])

Nunca use um objeto mutável como valor padrão. O objeto é criado uma vez, quando o `def` é executado, e seria compartilhado por todas as chamadas. Use `None` como sentinela.

In [ ]:
def add_message(role: str, content: str, history: list[dict] | None = None) -> list[dict]:
    """Acrescenta uma mensagem e devolve o histórico."""
    if history is None:
        history = []
    history.append({"role": role, "content": content})
    return history

print(add_message("user", "First"))
print(add_message("user", "Second"))

## Listas e dicionários como estado

Na disciplina, mensagens, memória, traço e resultados são listas e dicionários. As operações essenciais são acesso seguro com `get`, iteração por `items`, comprehensions, desempacotamento e construção de novos registros com `**`.

In [ ]:
trace = [
    {"kind": "model", "tokens": 28, "latency": 0.31},
    {"kind": "tool", "name": "search", "latency": 0.08},
    {"kind": "model", "tokens": 17, "latency": 0.22},
]

model_steps = [step for step in trace if step["kind"] == "model"]
total_tokens = sum(step.get("tokens", 0) for step in trace)
latencies = {index: step["latency"] for index, step in enumerate(trace)}
print(model_steps, total_tokens, latencies)

In [ ]:
config = {"temperature": 0.0, "max_tokens": 128}
experiment = {**config, "max_tokens": 64, "name": "short"}
first, *middle, last = ["plan", "search", "read", "answer"]
print(experiment)
print(first, middle, last)

Dicionários preservam ordem de inserção, mas isso não os transforma em um esquema. Antes de consumir dados vindos de um modelo, arquivo ou API, valide presença, tipo e domínio dos campos.

## Funções são valores

Uma função pode ser armazenada, passada como argumento e devolvida. É isso que permite entregar uma lista de ferramentas ao agente, escolher uma por nome e envolver uma função com um decorador. Parâmetros após `*` só podem ser informados pelo nome, o que torna chamadas extensas mais legíveis.

In [ ]:
def add(a: int, b: int) -> int:
    """Add two integer numbers."""
    return a + b

def multiply(a: int, b: int) -> int:
    """Multiply two integer numbers."""
    return a * b

tools = {fn.__name__: fn for fn in [add, multiply]}
call = {"name": "multiply", "arguments": {"a": 6, "b": 7}}
result = tools[call["name"]](**call["arguments"])
print(result)

In [ ]:
def invoke(fn: Callable[..., Any], /, *args: Any, retries: int = 0, **kwargs: Any) -> Any:
    """Executa uma função; `retries` só pode ser passado por nome."""
    for attempt in range(retries + 1):
        try:
            return fn(*args, **kwargs)
        except Exception:
            if attempt == retries:
                raise

print(invoke(add, 20, 22, retries=1))

`*args` reúne argumentos posicionais em uma tupla; `**kwargs` reúne argumentos nomeados em um dicionário. São úteis na infraestrutura que encaminha chamadas, mas uma ferramenta exposta ao modelo deve preferir uma assinatura explícita: ela é documentação e contrato.

## Closures e decoradores

Uma closure é uma função que conserva nomes do escopo em que foi criada. Um decorador recebe uma função e devolve outra. No `agentkit`, o decorador de ferramenta anexa metadados derivados da própria assinatura, evitando manter código e descrição em lugares separados.

In [ ]:
def limit_calls(max_calls: int):
    """Cria um decorador que limita o número de chamadas."""
    def decorate(fn):
        calls = 0

        @wraps(fn)
        def wrapped(*args, **kwargs):
            nonlocal calls
            if calls >= max_calls:
                raise RuntimeError("Call limit reached")
            calls += 1
            return fn(*args, **kwargs)
        return wrapped
    return decorate

@limit_calls(2)
def ping(text: str) -> str:
    """Return the received text."""
    return text

print(ping("one"), ping("two"), ping.__name__)

`wraps` preserva nome, docstring e anotações da função original. Sem isso, a introspecção enxergaria apenas `wrapped(*args, **kwargs)`, tornando o contrato da ferramenta inútil. `nonlocal` informa que a atribuição modifica o nome do escopo externo, e não cria uma variável local nova.

## Tipos como documentação executável

Anotações de tipo não são validadas automaticamente pelo Python. Elas documentam, alimentam verificadores estáticos e podem ser lidas em tempo de execução. `Literal` descreve um conjunto fechado; `TypedDict` documenta dicionários com campos conhecidos; uniões com `|` representam alternativas.

In [ ]:
class Message(TypedDict):
    role: Literal["system", "user", "assistant", "tool"]
    content: str

def recent_messages(history: list[Message], limit: int = 4) -> list[Message]:
    """Devolve no máximo as últimas `limit` mensagens."""
    if limit < 0:
        raise ValueError("limit must be non-negative")
    return history[-limit:] if limit else []

typed_history: list[Message] = [{"role": "user", "content": "Hello"}]
print(recent_messages(typed_history))
print(get_type_hints(recent_messages))

A anotação reduz ambiguidades, mas a fronteira do sistema ainda precisa de validação em tempo de execução. Nesta disciplina, Pydantic será usado quando a saída do modelo precisar obedecer a um esquema; para estruturas internas simples, listas e dicionários anotados mantêm o fluxo visível.

## Exceções e fronteiras de erro

Exceções interrompem o caminho normal e carregam contexto sobre a falha. Capture apenas os erros que você consegue tratar. Um `except Exception` amplo no laço principal pode transformar erro de programação em comportamento silencioso; ele faz sentido somente numa fronteira em que a falha será registrada e devolvida como observação explícita.

In [ ]:
def parse_tool_call(text: str) -> dict | None:
    """Interpreta uma chamada JSON ou sinaliza que o texto é resposta final."""
    text = text.strip()
    if not text.startswith("{"):
        return None
    try:
        call = json.loads(text)
    except json.JSONDecodeError as error:
        raise ValueError(f"Invalid tool call JSON: {error.msg}") from error
    if set(call) != {"name", "arguments"} or not isinstance(call["arguments"], dict):
        raise ValueError("Tool call must contain name and arguments")
    return call

for sample in ['Final answer', '{"name": "add", "arguments": {"a": 2, "b": 3}}']:
    print(parse_tool_call(sample))

In [ ]:
def run_tool(call: dict, registry: dict[str, Callable[..., Any]]) -> dict:
    """Executa a ferramenta e torna sucesso ou falha observável."""
    name = call["name"]
    if name not in registry:
        return {"ok": False, "error": f"Unknown tool: {name}"}
    try:
        value = registry[name](**call["arguments"])
        return {"ok": True, "value": value}
    except (TypeError, ValueError) as error:
        return {"ok": False, "error": str(error)}

print(run_tool({"name": "add", "arguments": {"a": 20, "b": 22}}, tools))
print(run_tool({"name": "missing", "arguments": {}}, tools))

## Iteráveis, iteradores e geradores

Um iterável pode produzir um iterador; um iterador entrega um item por vez até `StopIteration`. Geradores, escritos com `yield`, guardam seu ponto de execução entre itens. Eles são úteis para pipelines e eventos de um traço sem materializar tudo de uma vez. Não significam streaming do modelo: aqui são apenas um mecanismo da linguagem.

In [ ]:
def batched(items: Iterable[Any], size: int) -> Iterator[list[Any]]:
    """Agrupa itens em lotes de tamanho máximo `size`."""
    if size <= 0:
        raise ValueError("size must be positive")
    batch = []
    for item in items:
        batch.append(item)
        if len(batch) == size:
            yield batch
            batch = []
    if batch:
        yield batch

print(list(batched(range(7), size=3)))

Expressões geradoras usam parênteses e são preguiçosas; comprehensions de lista usam colchetes e materializam o resultado. Use uma lista quando o histórico precisa ser consultado novamente e um gerador quando os itens serão consumidos uma vez.

## Arquivos, caminhos e JSON

`Path` representa caminhos sem concatenação manual de strings. O bloco `with` é um gerenciador de contexto: garante a liberação do recurso mesmo quando ocorre uma exceção. Métodos como `read_text` e `write_text` já cuidam desse ciclo para arquivos simples. Quando um caminho vem do usuário ou do modelo, resolva-o e verifique a fronteira antes do acesso.

In [ ]:
def path_inside(root: Path, relative_path: str) -> Path:
    """Resolve um caminho e recusa destinos fora da raiz."""
    root = root.resolve()
    target = (root / relative_path).resolve()
    if not target.is_relative_to(root):
        raise ValueError("Path outside allowed directory")
    return target

with tempfile.TemporaryDirectory() as directory:
    root = Path(directory)
    target = path_inside(root, "memory/state.json")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(json.dumps({"facts": ["Python >= 3.11"]}, indent=2), encoding="utf-8")
    restored = json.loads(target.read_text(encoding="utf-8"))
    print(restored)

JSON só representa objetos, arrays, strings, números, booleanos e `null`. Tuplas voltam como listas, chaves de dicionários tornam-se strings e objetos Python arbitrários não são serializáveis. Essa limitação é útil: ela força o estado do agente a permanecer inspecionável.

## Introspecção: da função ao contrato

Introspecção permite examinar uma função durante a execução. Nome, docstring, assinatura e anotações já contêm quase tudo que o modelo precisa saber sobre uma ferramenta. O exemplo suporta deliberadamente apenas quatro tipos: limites explícitos são melhores que uma conversão que parece geral e falha silenciosamente.

In [ ]:
TYPE_NAMES = {str: "string", int: "integer", float: "number", bool: "boolean"}

def tool(fn: Callable[..., Any]) -> Callable[..., Any]:
    """Anexa um esquema simples à função e devolve a própria função."""
    description = inspect.getdoc(fn)
    if not description:
        raise ValueError(f"Tool {fn.__name__} needs a docstring")
    signature = inspect.signature(fn)
    hints = get_type_hints(fn)
    parameters = {}
    required = []
    for name, parameter in signature.parameters.items():
        annotation = hints.get(name)
        if annotation not in TYPE_NAMES:
            raise TypeError(f"Unsupported annotation for {name}: {annotation}")
        parameters[name] = {"type": TYPE_NAMES[annotation]}
        if parameter.default is inspect.Parameter.empty:
            required.append(name)
    fn.tool_schema = {
        "name": fn.__name__,
        "description": description.splitlines()[0],
        "parameters": parameters,
        "required": required,
    }
    return fn

In [ ]:
@tool
def search(query: str, limit: int = 3) -> str:
    """Search the course knowledge base."""
    return f"Found at most {limit} results for {query!r}."

print(search("memory"))
print(json.dumps(search.tool_schema, indent=2))

A docstring da ferramenta está em inglês porque entra no prompt do modelo. A docstring interna do decorador permanece em português. O decorador devolve a própria função: ela continua simples, chamável e testável sem um agente.

## Testes: comportamento antes de integração

Teste primeiro funções determinísticas. Um modelo falso torna o laço reproduzível e evita download de pesos. `assert` é suficiente para exemplos no notebook; no projeto, os mesmos casos devem ir para `pytest`.

In [ ]:
assert add(20, 22) == 42
assert parse_tool_call("A final answer") is None
assert path_inside(Path("/tmp/safe"), "notes.txt") == Path("/tmp/safe/notes.txt")

try:
    path_inside(Path("/tmp/safe"), "../outside.txt")
except ValueError:
    pass
else:
    raise AssertionError("Traversal should have been rejected")

print("All deterministic tests passed.")

## Integração: um laço de agente mínimo

Agora as peças se encontram. O modelo falso devolve respostas predeterminadas. O laço mantém mensagens, pede uma continuação, interpreta uma possível chamada, executa a ferramenta, registra a observação e repete até obter texto final ou atingir o limite. O limite faz parte da correção: sem ele, uma sequência de chamadas pode nunca terminar.

In [ ]:
class FakeLLM:
    """Modelo falso que devolve uma resposta por chamada."""
    def __init__(self, responses: Iterable[str]):
        self.responses = iter(responses)

    def chat(self, messages: list[dict]) -> str:
        return next(self.responses)

def run_agent(llm: Any, user_input: str, tool_functions: list[Callable], *, max_steps: int = 4) -> dict:
    """Executa o menor laço completo de um agente com ferramentas."""
    registry = {fn.__name__: fn for fn in tool_functions}
    messages = [{"role": "user", "content": user_input}]
    trace = []

    for step in range(max_steps):
        output = llm.chat(messages)
        messages.append({"role": "assistant", "content": output})
        trace.append({"step": step, "kind": "model", "output": output})
        call = parse_tool_call(output)
        if call is None:
            return {"answer": output, "messages": messages, "trace": trace}

        observation = run_tool(call, registry)
        content = json.dumps(observation, ensure_ascii=False)
        messages.append({"role": "tool", "content": content})
        trace.append({"step": step, "kind": "tool", "name": call["name"], **observation})

    raise RuntimeError(f"Agent exceeded {max_steps} steps")

In [ ]:
fake_llm = FakeLLM([
    '{"name": "add", "arguments": {"a": 20, "b": 22}}',
    "The result is 42.",
])
agent_result = run_agent(fake_llm, "Use the add tool to calculate 20 + 22.", [add])
print(agent_result["answer"])
print(json.dumps(agent_result["trace"], indent=2))

Observe a separação de responsabilidades. `parse_tool_call` interpreta texto; `run_tool` executa código; `run_agent` coordena o estado. Cada função pode ser testada isoladamente. Classes aparecem apenas onde há identidade e estado persistente — o modelo falso guarda a posição nas respostas — e não como embalagem automática de toda função.

## Concorrência: o mínimo necessário

Mais adiante, ferramentas independentes podem ser executadas concorrentemente. `async def` cria uma coroutine, `await` suspende a coroutine enquanto aguarda outra operação, e `asyncio.gather` reúne resultados preservando a ordem das entradas. Isso beneficia tarefas limitadas por espera, como rede, e não torna computação pesada mais rápida. O laço básico da Unidade I permanece síncrono para que seu controle seja visível.

In [ ]:
import asyncio

async def fetch_fake(name: str, delay: float) -> dict:
    """Simula uma operação de entrada e saída."""
    await asyncio.sleep(delay)
    return {"name": name, "ok": True}

async def fetch_all() -> list[dict]:
    return await asyncio.gather(
        fetch_fake("documentation", 0.01),
        fetch_fake("database", 0.01),
    )

await fetch_all()

## Exercícios

Os exercícios seguem o padrão da disciplina: há um comportamento verificável, um caso de falha e uma extensão curta. Resolva sem alterar as células anteriores.

### Estado sem efeitos colaterais

Implemente `append_message(history, role, content)` de modo que devolva um novo histórico e não altere a lista nem os dicionários recebidos. Verifique identidade com `is` e provoque uma alteração no resultado para demonstrar independência.

In [ ]:
def append_message(history: list[dict], role: str, content: str) -> list[dict]:
    # TODO: devolva um novo histórico sem compartilhar dicionários mutáveis
    raise NotImplementedError

# original = [{"role": "user", "content": "Hi"}]
# updated = append_message(original, "assistant", "Hello")
# assert len(original) == 1 and len(updated) == 2

### Memória por relevância lexical

Implemente `recall(memory, query, limit=3)`. Cada item da memória é um dicionário com `text`. Conte quantas palavras da consulta aparecem no texto, ignore maiúsculas e devolva os itens de maior pontuação. Itens com pontuação zero não entram. Decida e documente como empates são resolvidos.

In [ ]:
def recall(memory: list[dict], query: str, limit: int = 3) -> list[dict]:
    # TODO: calcule, filtre, ordene e limite
    raise NotImplementedError

memory = [
    {"text": "Agents use tools to act."},
    {"text": "Memory stores useful facts."},
    {"text": "Tools are Python functions."},
]
# assert recall(memory, "Python tools", limit=1) == [memory[2]]

### Contrato de ferramenta

Estenda o decorador `tool` para exigir anotação do retorno, recusar parâmetros variádicos e incluir valores padrão no esquema. Escreva três testes: uma função válida, uma sem docstring e uma com `*args`.

### Laço tolerante a falhas

Altere `run_agent` para que JSON de chamada inválido seja convertido em uma mensagem `tool` com o erro, permitindo ao modelo tentar novamente. O laço ainda deve parar após `max_steps`. Use um `FakeLLM` que primeiro produza JSON inválido, depois uma chamada correta e por fim a resposta.

### Harness de avaliação

Crie `evaluate(agent, cases)`, em que cada caso contém `input` e `expected`. Devolva um dicionário com total, acertos, taxa de acerto e os casos que falharam. Depois acrescente ao resultado o número de passos do traço. Não use um modelo real no teste.

## Checklist para os próximos notebooks

Ao ler código da disciplina, confirme: qual é a forma dos dados; quem pode mutá-los; qual função transforma cada etapa; onde dados externos são validados; quais exceções são tratáveis; qual é a condição de parada; e que parte pode ser testada sem carregar um modelo. Se essas respostas estiverem explícitas, o Python deixou de ser obstáculo e passou a revelar a arquitetura do agente.